# 실습 - 이미지 추론 (분류 · 탐지 · 세그멘테이션)

**pipeline으로 세 가지 작업 수행**

- pipeline — 사전학습 모델을 추론에 사용하는 표준 도구
- 지정 항목 — 작업(task) · 모델 · 입력
- 작업 이름만 변경하면 분류 · 탐지 · 세그멘테이션 모두 동일한 방식
- 모델 파라미터는 변경되지 않음 (추론 전용)

**실습 순서**

- 환경 준비
- 이미지 불러오기
- 작업 ① 이미지 분류
- 작업 ② 객체 탐지
- 작업 ③ 세그멘테이션
- 정리

---
## 0. 환경 준비

- `transformers` — 사전학습 모델로 추론 수행
- `PIL` — 이미지 열기 · 표시 · 그리기
- `requests` — 웹 이미지 받아오기

In [ ]:
from transformers import pipeline
from PIL import Image, ImageDraw
import requests

print("준비 완료")

**경고 메시지 안내**

- 실행 시 붉은 경고 문구 출력 가능
- 경고(Warning)와 오류(Error)는 구분 — 결과가 출력되면 정상
- `This is expected` 문구가 포함된 경우 무시 가능

---
## 1. 이미지 불러오기

**세 가지 방법**

- 직접 업로드 — `Image.open("파일경로")`
- URL 입력 — 주소에서 받아오기
- Hub 데이터셋 — `load_dataset`

**실습 진행**

- 아래 코드는 직접 업로드 방식 사용
- 본인 이미지 사용 시 `Image.open("파일명.jpg")` 로 교체
- 화면에 이미지가 표시되면 준비 완료

In [ ]:
image = Image.open("cat.jpg")
image

In [ ]:
print("크기:", image.size)
print("색상 방식:", image.mode)

---
## 2. 작업 ① 이미지 분류

**개요**

- 작업 이름 — `image-classification`
- 수행 내용 — 이미지 전체가 무엇인지 판별
- 출력 형태 — 라벨(label)과 점수(score)의 목록

### 2-1. 모델 준비

- `task` — 수행할 작업 이름
- `model` — Hub에서 선택한 모델 (`제작자/모델명`)
- 첫 실행 시 모델 내려받기로 시간 소요

In [ ]:
clf = pipeline(
    task="image-classification",
    model="google/vit-base-patch16-224"
)

### 2-2. 분류 실행

- 입력 — 파일 경로 · URL · 이미지 객체 모두 가능
- 출력 — 점수가 높은 순으로 정렬된 후보 목록
- 기본 5개 후보 반환

In [ ]:
result_clf = clf(image)
result_clf

### 2-3. 결과 읽기

- **label** — 모델이 예측한 분류 이름
- **score** — 해당 분류에 대한 예측 확률 (1에 가까울수록 확신이 큼)
- 상위 후보가 함께 제시됨 — 단일 정답이 아님
- 상위 후보가 서로 유사하면 모델이 판단을 나눈 상태

In [ ]:
for r in result_clf:
    print(r["label"], "-", round(r["score"], 4))

### 2-4. 출력 개수 조절

- `top_k` — 반환할 후보 수 지정

In [ ]:
clf(image, top_k=3)

---
## 3. 작업 ② 객체 탐지

**개요**

- 작업 이름 — `object-detection`
- 수행 내용 — 대상의 종류와 **위치** 판별
- 출력 형태 — 라벨 · 점수 · 위치 상자(box)
- 앞 작업과 동일한 구조 — 작업 이름과 모델만 변경

### 3-1. 모델 준비

In [ ]:
det = pipeline(
    task="object-detection",
    model="facebook/detr-resnet-50"
)

### 3-2. 탐지 실행

- 이미지 속 대상마다 하나씩 출력
- **box** — 대상의 위치 (사각형 좌표)
- 좌표 기준 — 이미지 좌측 상단이 (0, 0)

In [ ]:
result_det = det(image)
result_det

### 3-3. 결과 읽기

- 탐지된 대상 수 확인
- 라벨과 점수는 분류와 동일한 형식
- 새로 추가된 항목은 **box** 하나뿐

In [ ]:
print("탐지된 대상 수:", len(result_det))

for r in result_det:
    print(r["label"], "-", round(r["score"], 3), "-", r["box"])

### 3-4. 결과 시각화

- 좌표를 사각형으로 그려 위치 확인
- `image.copy()` — 원본 보존을 위한 사본 생성
- `ImageDraw.Draw` — 이미지에 그림을 그리는 도구
- 코드는 결과 확인용 — 이해보다 실행에 중점

In [ ]:
draw_image = image.copy()
draw = ImageDraw.Draw(draw_image)

for r in result_det:
    box = r["box"]
    draw.rectangle(
        (box["xmin"], box["ymin"], box["xmax"], box["ymax"]),
        outline="red", width=3
    )
    draw.text((box["xmin"], box["ymin"] - 12), r["label"], fill="red")

draw_image

### 3-5. 낮은 점수 제외

- 점수가 낮은 항목은 부정확한 상자를 생성
- 기준값 이상만 선택하여 다시 확인

In [ ]:
draw_image = image.copy()
draw = ImageDraw.Draw(draw_image)

for r in result_det:
    if r["score"] < 0.9:
        continue
    box = r["box"]
    draw.rectangle(
        (box["xmin"], box["ymin"], box["xmax"], box["ymax"]),
        outline="red", width=3
    )
    draw.text((box["xmin"], box["ymin"] - 12), r["label"], fill="red")

draw_image

---
## 4. 작업 ③ 세그멘테이션

**개요**

- 작업 이름 — `image-segmentation`
- 수행 내용 — 픽셀 단위로 영역 구분
- 출력 형태 — 라벨 · 점수 · 마스크(mask)
- 좌표가 아닌 **영역 자체**를 반환

### 4-1. 모델 준비

In [ ]:
seg = pipeline(
    task="image-segmentation",
    model="facebook/detr-resnet-50-panoptic"
)

### 4-2. 분할 실행

- 구분된 영역마다 하나씩 출력
- **mask** — 해당 영역을 표시한 흑백 이미지
- 배경까지 구분되어 항목 수가 여러 개일 수 있음

In [ ]:
result_seg = seg(image)

print("구분된 영역 수:", len(result_seg))

for r in result_seg:
    score = r["score"]
    print(r["label"], "-", round(score, 3) if score else "-")

### 4-3. 마스크 확인

- 대상 영역은 흰색, 나머지는 검은색
- `result_seg[0]["mask"]` — 첫 번째 마스크
- 번호를 바꾸면 다른 영역 확인 가능

In [ ]:
result_seg[0]["mask"]

### 4-4. 원본에 겹치기

- `convert("RGBA")` — 투명도 처리가 가능한 형식으로 변환
- `Image.new` — 반투명 색 판 생성
- `paste(..., mask)` — 마스크 영역에만 색 입히기

In [ ]:
overlay = image.convert("RGBA")
mask = result_seg[0]["mask"]

color = Image.new("RGBA", image.size, (255, 0, 0, 100))
overlay.paste(color, (0, 0), mask)

overlay

---
## 5. 정리

**세 가지 작업의 공통점**

- 코드 구조 동일 — `pipeline(task=..., model=...)` 후 이미지 입력
- 변경 항목 — 작업 이름과 모델뿐
- 모델 파라미터는 변경되지 않음 (추론 전용)

**출력 형태의 차이**

| 작업 | 작업 이름 | 출력 |
|---|---|---|
| 이미지 분류 | `image-classification` | 라벨 · 점수 |
| 객체 탐지 | `object-detection` | 라벨 · 점수 · 위치(box) |
| 세그멘테이션 | `image-segmentation` | 라벨 · 점수 · 영역(mask) |

**직접 해보기**

- 본인 이미지로 세 작업 모두 실행
- 각 작업의 출력 형태 비교
- 결과가 기대와 일치하는지 확인
- 모델을 생략하면 작업별 기본 모델 자동 준비

---

출처: Hugging Face Transformers 공식 문서 (Pipeline)